# PRAGMA Phase 2 - Point-in-time evaluation records

Builds `EvaluationRecord`s from the Phase 1 synthetic corpus using `PointInTimeRecordBuilder` (`src/pragma/data/records.py`), and inspects them for the leakage/correctness properties required by section 5.4 of `plans/PRAGMA-Implementation-Plan.md` and ADR 0002.

Per ADR 0008, the MVP samples exactly **one evaluation point per entity** (the latest observed event, or `signup_at` for zero-event entities) and assigns train/val/test splits deterministically by hashing `entity_id`. Run `000_synthetic_data_generation.ipynb` (or `scripts/generate_synthetic_data.py`) first so `data/raw/` exists.

This notebook is a thin, inspectable wrapper: all record-building logic lives in `src/pragma/`, so it is unit-tested independently of this notebook (`tests/unit/test_records.py`).

In [1]:
from pathlib import Path

import pandas as pd

from pragma.data.records import PointInTimeRecordBuilder, SplitConfig
from pragma.schema import SchemaRegistry

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = REPO_ROOT / "data" / "raw"
RAW_DIR

WindowsPath('C:/Users/levyr/Desktop/random-projects/pragma/data/raw')

## Load the raw corpus and build evaluation records

In [2]:
events_df = pd.read_parquet(RAW_DIR / "events.parquet")
profile_df = pd.read_parquet(RAW_DIR / "profile_state.parquet")

registry = SchemaRegistry.default()
builder = PointInTimeRecordBuilder(registry, SplitConfig())
records = builder.build(events_df, profile_df)

print(f"{len(records)} evaluation records for {len(profile_df)} entities (one record per entity, per ADR 0008)")

500 evaluation records for 500 entities (one record per entity, per ADR 0008)


## Split distribution

Deterministic by `entity_id` hash (`SplitConfig`, default 80/10/10). Re-running the cell above reproduces identical assignments.

In [3]:
split_counts = pd.Series([r.split for r in records]).value_counts()
split_counts.rename("n_records").to_frame()

,n_records
train,399
test,53
val,48


## History-length distribution by edge-case cohort

Sanity check against the synthetic generator's deliberate edge cases (zero-event, single-event, long-history, same-timestamp entities).

In [4]:
n_events = pd.Series(
    {r.entity_id: len(r.events_before_evaluation) for r in records}, name="n_events"
)
cohort = profile_df.set_index("entity_id")["_edge_case_profile"]
summary = pd.concat([n_events, cohort], axis=1)
summary.groupby("_edge_case_profile")["n_events"].describe()

,count,mean,std,min,25%,50%,75%,max
_edge_case_profile,,,,,,,,
long_history,5.0,3000.000000,0.000000,3000.0,3000.0,3000.0,3000.00,3000.0
normal,455.0,199.428571,115.209323,2.0,96.5,197.0,295.50,399.0
same_timestamp,10.0,226.700000,129.101209,96.0,131.5,147.0,366.25,398.0
single_event,15.0,1.000000,0.000000,1.0,1.0,1.0,1.00,1.0
zero_events,15.0,0.000000,0.000000,0.0,0.0,0.0,0.00,0.0


## Leakage checks

These are the correctness properties ADR 0002 and section 5.4 require: no input event occurs after its record's `evaluation_time`, and every visible milestone timestamp is `<= evaluation_time`. `tests/unit/test_records.py` checks these on every commit; this cell is the same check made visible.

In [5]:
violations = [
    (r.entity_id, e.event_id)
    for r in records
    for e in r.events_before_evaluation
    if e.created_at > r.evaluation_time
]
milestone_violations = [
    (r.entity_id, key)
    for r in records
    for key, ts in r.profile_state.milestones.items()
    if ts is not None and ts > r.evaluation_time
]
print(f"event-after-evaluation-time violations: {len(violations)}")
print(f"future-milestone violations: {len(milestone_violations)}")
assert not violations and not milestone_violations

event-after-evaluation-time violations: 0
future-milestone violations: 0


## Inspect one record from each interesting cohort

A zero-event entity (profile state only, no event history) and a normal entity with a non-trivial history and at least one resolved lifelong milestone.

In [6]:
records_by_entity = {r.entity_id: r for r in records}

zero_event_entity = cohort[cohort == "zero_events"].index[0]
zero_record = records_by_entity[zero_event_entity]
print("entity:", zero_record.entity_id)
print("evaluation_time:", zero_record.evaluation_time)
print("n events:", len(zero_record.events_before_evaluation))
print("attributes:", zero_record.profile_state.attributes)
print("milestones:", zero_record.profile_state.milestones)

entity: u000043
evaluation_time: 2024-10-18 21:58:22+00:00
n events: 0
attributes: {'plan': 'premium', 'balance_quantile': 'q3', 'kyc_level': 'full', 'age_band': '18-24', 'is_active': True, 'country': 'IE'}
milestones: {'first_card_payment_at': None, 'first_topup_at': None, 'signup_at': Timestamp('2024-10-18 21:58:22+0000', tz='UTC'), 'first_p2p_at': None}


In [7]:
normal_entities = cohort[cohort == "normal"].index
normal_record = next(
    records_by_entity[e]
    for e in normal_entities
    if any(v is not None for v in records_by_entity[e].profile_state.milestones.values())
)
print("entity:", normal_record.entity_id)
print("evaluation_time:", normal_record.evaluation_time)
print("n events:", len(normal_record.events_before_evaluation))
print("milestones:", normal_record.profile_state.milestones)
print("first 3 events:")
for e in normal_record.events_before_evaluation[:3]:
    print(" ", e.created_at, e.event_type, e.fields)

entity: u000000
evaluation_time: 2026-06-29 09:06:21+00:00
n events: 219
milestones: {'first_card_payment_at': Timestamp('2024-02-29 19:13:03+0000', tz='UTC'), 'first_topup_at': Timestamp('2024-02-28 23:18:16+0000', tz='UTC'), 'signup_at': Timestamp('2024-02-23 15:33:54+0000', tz='UTC'), 'first_p2p_at': Timestamp('2024-05-01 01:51:05+0000', tz='UTC')}
first 3 events:
  2024-02-28 23:18:16+00:00 topup {'amount': 35.68, 'currency': 'GBP', 'direction': 'in', 'description': 'dinner with friends', 'channel': 'open_banking'}
  2024-02-29 19:13:03+00:00 card_payment {'amount': 3.44, 'currency': 'EUR', 'direction': 'out', 'description': 'metal plan', 'mcc': '5411', 'merchant_name': 'Tesco'}
  2024-03-03 23:18:10+00:00 card_payment {'amount': 23.45, 'currency': 'PLN', 'direction': 'out', 'description': 'dinner with friends', 'mcc': '5812', 'merchant_name': 'Amazon'}
